Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [ ]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [ ]:
import os
import json
import copy
import pycomet
import numpy as np
import scipy as sp

from scipy.linalg import expm
from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from hypernet.thermophysics import specie_thermo as st_mod

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x=None,
    y=None,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    n, d = y.shape if (x is not None) else yref.shape
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=n//20,
        ls='--',
        lw=1
    )
    for i in range(d):
        if (x is not None):
            ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
        if (xref is not None):
            ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
        ax.plot([], [], c=COLORS[i], label=names[i])
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [ ]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [ ]:
def get_data(solution):
    t, T = [solution[k] for k in ('t', 'T')]
    na, nm = [solution[f"X_{s}"] * solution['n'] for s in sp_th.keys()]
    p = solution['p'][0].squeeze()
    return t, nm, T, p

In [ ]:
cases = ["p1k.T1k.O05", "p5k.T1k.O05", "p1k.T5k.O05", "p1k.T1k.O10"]
sol = {k: h5todict(h5file=f'./../physics/solution.{k}.hdf5') for k in cases}
solref = sol["p1k.T1k.O05"]

In [ ]:
t, nm, T, p = get_data(solref)
st_mod.update_specie_thermos(sp_th, T)

In [ ]:
plot_var(
    xref=t.squeeze()[1:],
    yref=nm[1:],
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(nm.shape[1])],
    figname=path_to_figs + "/solution"
)

\begin{align}
    \frac{n_i(t)}{n}
    & =\frac{q_i(t)}{Q} \\
    & =\frac{q_i^\text{eq}f(t)}{Q} \\
    & =\frac{q_i^\text{eq}f(t)}{\sum_iq_i^\text{eq}} \\
    & =\frac{q_i^\text{eq}}{\sum_iq_i^\text{eq}}\exp\left(-\frac{\delta e_i(t)}{k_\text{B}T}\right)
\end{align}

\begin{equation}
    \delta e_i(t) = - k_\text{B}T\log\left(\frac{n_i(t)}{n}\frac{\sum_iq_i^\text{eq}}{q_i^\text{eq}}\right)
\end{equation}

In [ ]:
def get_energy(nm, T, stm):
    Qg = stm.pf.int.Q.numpy()
    Q = np.sum(Qg, axis=-1, keepdims=True)
    n = np.sum(nm, axis=-1, keepdims=True)
    de = -const.UKB*T*np.log((nm/n)*(Q/Qg))
    e_eq = stm.thermo.int.e.numpy() * stm.specie.m
    en = (e_eq + de) * nm
    en_eq = e_eq * nm
    return de, en, en_eq

In [ ]:
de, en, en_eq = get_energy(nm, T, sp_th["O2"])

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=de[1:]/const.eV_to_J,
    labels=[r'$t$ [s]', r'$\delta e$ [eV]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(de.shape[1])],
    figname=path_to_figs + "/denergy"
)

Decoupled Landau-Teller relaxation model
\begin{equation}
    \frac{de_i(t)}{dt}=-\frac{e_i(t)-e_i^{\mathrm{eq}}n_i^{\mathrm{eq}}}{\tau_i}
\end{equation}
with
\begin{equation}
    e_i(t) = \left[e_i^{\mathrm{eq}} + \delta e_i(t)\right]n_i(t)
\end{equation}
and
\begin{equation}
    \tau_i =\frac{f(T\,|\,\theta_i)}{p}
\end{equation}

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=en[1:],
    xref=t.squeeze()[1:],
    yref=en_eq[1:],
    labels=[r'$t$ [s]', r'$e$ [J m$^{-3}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(de.shape[1])],
    figname=path_to_figs + "/energy"
)

In [ ]:
idx = np.argmin(np.absolute(en - 0.6321*en_eq[-1]), axis=0)
taup = [float(t[i+1])*p for i in idx]
taup[0] = taup[0] * 5e-2
taup

Testing

In [ ]:
def integr(y0, yf, taup, p):
    res = sp.integrate.solve_ivp(
        fun=lambda t,y,taup,p: -(p/taup)*(y-yf),
        t_span=[t[0],t[-1]],
        y0=y0,
        method='LSODA',
        t_eval=t,
        args=(taup,p),
        atol=1e-9,
        rtol=1e-7
    )
    return res.y.reshape(-1,1)

In [ ]:
for case in cases:

    print(case)
    
    t, nm, T, p = get_data(sol[case])
    st_mod.update_specie_thermos(sp_th, T)
    de, en, en_eq = get_energy(nm, T, sp_th["O2"])

    en_pred = []
    t = t.squeeze()
    for i in range(3):
        y0 = en[0,i].reshape(-1)
        yf = en_eq[-1,i].reshape(-1)
        ei = integr(y0, yf, taup[i], p)
        en_pred.append(ei)
    en_pred = np.hstack(en_pred)

    plot_var(
        x=t.squeeze()[1:],
        y=en_pred[1:],
        xref=t.squeeze()[1:],
        yref=en[1:],
        labels=[r'$t$ [s]', r'$e$ [J m$^{-3}$]'],
        scales=['log', 'linear'],
        names=[r'O$_2^{(%d)}$' % (i+1) for i in range(de.shape[1])],
        figname=path_to_figs + f"/energy.{case}.png"
    )